# HiC-ECC | Module 1: Enhance (DeepHiC)
Convert HiC-Pro matrices → DeepHiC enhancement → .cool files.

## Config
**Edit only this cell.**

In [ ]:
# Tissues
TISSUES = [
    'Brain',
    'Kidney',
    'Large_Intestine',
    'Liver',
    'Lung',
    'Pancreas',
    'Small_Intestine',
    'Spleen',
]

# Resolution & genome
RESOLUTION  = 100000          # e.g. 10000 or 100000
RES_STR     = str(RESOLUTION) # used in filenames
GENOME      = 'mm10'
SPECIES     = 'mouse'         # passed to data_generate.py

# Paths
SAMPLE_DIR   = '/path/to/HiC_Pro_output'          # contains {tissue}_{res}_abs.bed and .matrix
DEEPHIC_ROOT = f'/path/to/DeepHiC/{GENOME}'       # DeepHiC working dir; mat/ and predict/ created here
COOL_DIR     = f'/path/to/cool_files/{RES_STR}'   # output cool files go here
CHROM_SIZES  = '/path/to/mm10.chrom.sizes'
CHECKPOINT   = '/path/to/DeepHiC/save/deephic_raw_16.pth'
DEEPHIC_PY   = '~/DeepHiC'                        # directory containing the DeepHiC scripts

# DeepHiC data_generate params
CHUNK  = 100
STRIDE = 100
BOUND  = 9999
LRC    = 100    # low-res cutoff

## Setup

In [ ]:
import os, subprocess
import numpy as np

os.makedirs(f'{DEEPHIC_ROOT}/mat', exist_ok=True)
print('Ready.')


# Section 1 - HiC-Pro → DeepHiC → predict

In [ ]:
for tissue in TISSUES:
    print(f'\n[{tissue}] hicpro2deephic')
    subprocess.run([
        'python', f'{DEEPHIC_PY}/hicpro2deephic.py',
        '--bed', f'{SAMPLE_DIR}/{tissue}_{RES_STR}_abs.bed',
        '--mat', f'{SAMPLE_DIR}/{tissue}_{RES_STR}.matrix',
        '-r', RES_STR,
        '-o', f'{DEEPHIC_ROOT}/mat/{tissue}_{RES_STR}',
    ], check=True)

    print(f'[{tissue}] data_generate')
    subprocess.run([
        'python', f'{DEEPHIC_PY}/data_generate.py',
        '-hr', RES_STR, '-lr', RES_STR,
        '-lrc', str(LRC), '-s', SPECIES,
        '-chunk', str(CHUNK), '-stride', str(STRIDE),
        '-bound', str(BOUND), '-scale', '1',
        '-c', f'{tissue}_{RES_STR}',
    ], check=True, cwd=DEEPHIC_ROOT)

    print(f'[{tissue}] data_predict')
    subprocess.run([
        'python', f'{DEEPHIC_PY}/data_predict.py',
        '-lr', RES_STR,
        '-ckpt', CHECKPOINT,
        '-c', f'{tissue}_{RES_STR}',
    ], check=True, cwd=DEEPHIC_ROOT)

    print(f'[{tissue}] done')

print('\nAll tissues enhanced.')

# Section 2 - Convert predictions to .cool

In [ ]:
RES_KB    = RESOLUTION // 1000
PREDICT_DIR = f'{DEEPHIC_ROOT}/predict'

for tissue in TISSUES:
    print(f'\n[{tissue}] building bg2')
    tissue_cool_dir = f'{COOL_DIR}/{tissue}'
    os.makedirs(tissue_cool_dir, exist_ok=True)

    bg2  = f'{tissue_cool_dir}/{tissue}_deephic_{RES_KB}kb.bg2'
    cool = f'{tissue_cool_dir}/{tissue}_deephic.{RES_KB}kb.cool'
    sr_dir = f'{PREDICT_DIR}/Mouse_{tissue}_{RESOLUTION}/sr16'

    # npz → bg2
    with open(bg2, 'w') as out:
        for fname in sorted(os.listdir(sr_dir)):
            if not fname.endswith(f'_{RESOLUTION}.npz'):
                continue
            chrom = fname.replace('predict_', '').replace(f'_{RESOLUTION}.npz', '')
            matrix = np.load(os.path.join(sr_dir, fname))['deephic']
            for i in range(matrix.shape[0]):
                for j in range(i, matrix.shape[0]):
                    if matrix[i, j] > 0:
                        out.write(f'{chrom}\t{i*RESOLUTION}\t{(i+1)*RESOLUTION}\t'
                                  f'{chrom}\t{j*RESOLUTION}\t{(j+1)*RESOLUTION}\t{matrix[i,j]}\n')

    # bg2 → cool
    print(f'[{tissue}] cooler load')
    subprocess.run([
        'cooler', 'load', '-f', 'bg2',
        f'{CHROM_SIZES}:{RESOLUTION}',
        bg2, cool,
        '--count-as-float', '--input-copy-status', 'duplex',
    ], check=True)

    print(f'[{tissue}] cooler balance')
    subprocess.run(['cooler', 'balance', cool, '--force'], check=True)

    print(f'[{tissue}] done → {cool}')

print('\nAll cool files ready.')